## Clarifying Questions

## Functional and Non Functional Requirements

## Core Entities

## Create a vehicle 
    - different options

In [ ]:
# option A ---> simple class 
class Vehicle:
    def __init__(self, category, plate):
        self.category = category
        self.plate = plate




In [ ]:
# option B --> Create a vehicle Abstraction and use Inheritance
from abc import ABC, abstractmethod

class Vehicle(ABC):

    @abstractmethod
    def required_size(self) -> SpotSize:
        pass


class Car(Vehicle):
    def required_size(self):
        pass

class Bike(Vehicle):
    def required_size(self):
        pass

class Truck(Vehicle):
    def required_size(self):
        pass
   
class SpotSize:
    pass
    

In [ ]:
## option C --> Composition

class VehicleType:
    def __init__(self, name):
        self.name = name


class Vehicle:
    def __init__(self, plate, category: VehicleType):
        self.plate = plate
        self.category = category

## Create SpotSize

In [6]:
from enum import Enum
from abc import ABC, abstractmethod

class SpotSize(Enum):
    '''
    Discrete Spot sizes
    
    '''

    SMALL = 1 # fit only bikes
    MEDIUM =  2 # fit bikes and car
    LARGE = 3 # fir bikes , car and truck


class Vehicle(ABC):
    
    def __init__(self, license_plate: str):
        # license_plate is the vehicle's identity in our system.
        # In a real system this would also have validation (length, format).
        self.license_plate = license_plate

    @property
    @abstractmethod
    def required_size(self) -> SpotSize:
        """
        The smallest spot size that can hold this vehicle.
        Used by the spot-finding algorithm to filter candidates.
        """
        ...

    def __repr__(self):
        # Useful for logging and debugging.
        return f"{self.__class__.__name__}({self.license_plate})"

class Bike(Vehicle):
    @property
    def required_size(self) -> SpotSize:
        return SpotSize.SMALL


class Car(Vehicle):
    @property
    def required_size(self) -> SpotSize:
        return SpotSize.MEDIUM


class Truck(Vehicle):
    @property
    def required_size(self) -> SpotSize:
        return SpotSize.LARGE
    

## Parking Spot
    - SpotSize
    - park
    - vacate
    - can_fit
    - status  (empty, occupied)

In [7]:
from typing import Optional

class ParkingSpot:

    def __init__(self, spot_id: str, size: SpotSize):
         self.spot_id = spot_id
         self.size = size

         self.vehicle: Optional[Vehicle] = None
         # self.occupied = False

    @property
    def is_free(self)->bool:
        return self.vehicle is None

    def can_fit(self, vehicle: Vehicle):
        '''
        A spot can fit a vehicle if 
        (1) it's currently free AND
        (2) it's at least as large as the vehicle requires.
        '''
        return self.is_free and self.size.value >= vehicle.required_size.value

    def vacate(self) -> Vehicle:
        if self.is_free:
            raise ValueError(f"Spot {self.spot_id} was already empty")
        vehicle = self.vehicle
        self.vehicle = None

        return Vehicle

    def __repr__(self):
        status = f"occupied by {self.vehicle.license_plate}" if not self.is_free else "free"
        return f"Spot({self.spot_id}, size={self.size.name}, {status})"   
        
    def park(self, vehicle: Vehicle) -> None:
        '''
        first check if the spot is empty
        if spot is empty --> check if the vehicle fits in the spot
        if it fits --> assign the vehicle to the spot and change the status from empty to occupied
        '''

        # first check if the spot is empty
        if not self.is_free:
            raise ValueError(
                f"Spot {self.spot_id} is already occupied by "
                f"{self.vehicle.license_plate}"
            )

        # if spot is empty --> check if the vehicle fits in the spot
        if not self.can_fit(vehicle):
            raise ValueError(
                f"Vehicle {vehicle.license_plate} (needs {vehicle.required_size.name}) "
                f"doesn't fit in spot {self.spot_id} (size {self.size.name})"
            )

        self.vehicle = vehicle


## Floor

    - floor number 
    - spots --> List(ParkingSpot)
    - display board
    

In [62]:
class Floor:
    
    def __init__(self, floor_number: int, spots: list[ParkingSpot]):
        self.floor_number = floor_number
        self.spots = spots

        self._observers: list["FloorObserver"] = []


    ## attach a display board to the floor

    def attach(self, observer: "FloorObserver") -> None:
        """Register an observer to receive update notifications."""
        self._observers.append(observer)

        
    ## remove a display from the floor
    def detach(self, observer: "FloorObserver") -> None:
        """Unregister an observer. Useful for tests and dynamic UIs."""
        self._observers.remove(observer)
    

    def _notify(self) -> None:
        '''
        Internal: tell all observers this floor's state changed.
        Called after every park/vacate. We don't pass any data —
        observers can pull what they need from the floor.
        '''
        for observer in self._observers:
            observer.update(self)
        

    def find_free_spot(self, vehicle:Vehicle) -> Optional[ParkingSpot]:

        '''
        Strategy  : Find the smallest free spot that can hold the vehicle
        - Linear Scan (Time Complexity O(n))
        
        '''
        # Filter to candidate spots.
        candidates = [s for s in self.spots if s.can_fit(vehicle)]

        if not candidates:
            return None
                                                         
        return min(candidates, key=lambda s: s.size.value)


    def vacate(self, spot: ParkingSpot) -> Vehicle:
        vehicle = spot.vacate()
        self._notify()
        return vehicle

    def free_spots_by_size(self) -> dict[SpotSize, int]:
        '''
        Aggregate count of free spots per size category.
        This is what DisplayBoard renders.
        '''
        counts = {size: 0 for size in SpotSize}
        for spot in self.spots:
            if spot.is_free:
                counts[spot.size] += 1
        return counts

    def park(self, vehicle:Vehicle) -> Optional[ParkingSpot]:
        spot = self.find_free_spot(vehicle)
        if spot is None:
            return None

        spot.park(vehicle)  ## there are redundant checks here
        self._notify()
        return spot

    def __repr__(self):
        return f"Floor({self.floor_number}, {len(self.spots)} spots)"

    

## Ticket


In [63]:
from dataclasses import dataclass, field
from datetime import datetime
import uuid


@dataclass
class Ticket:
    '''
    A receipt issued at entry, redeemed at exit.
    '''
    
    vehicle: Vehicle
    spot: ParkingSpot
    floor_number: int
    entry_time:datetime = field(default_factory=datetime.now)
    exit_time:Optional[datetime] = None
    amount_paid: Optional[float] = None
    id: str= field(default_factory=lambda: str(uuid.uuid4()))
    

## Entry gate and exit gate

     - Responsibility of entry gate
           - attribute - (gate no, parking lot)
           - behaviour - grant entry to a vehicle ( issue a ticket )

    - Responsibility of exit gate
        - calculate the amount based on the pricing strategy

In [64]:
class EntryGate:

    def __init__(self, gate_id: str, lot: "ParkingLot"):
        self.gate_id = gate_id
        self.lot = lot

    def admit(self, vehicle:Vehicle):
        '''
        Admit a vehicle and issue a ticket
        '''
        ## defer the park to the lot
        ticket = self.lot.park(vehicle)

        if ticket is None:
            print(f"[Gate {self.gate_id}] Sorry, no space for {vehicle}")
        else:
            print(f"[Gate {self.gate_id}] Issued ticket {ticket.id} to {vehicle}")

        return ticket


class ExitGate:
    def __init__(self, gate_id: str, lot: "ParkingLot", pricing: "PricingStrategy"):
        self.gate_id = gate_id
        self.lot = lot
        self.pricing = pricing

    def process(self, ticket:Ticket) -> float:
        '''
        - stamp the exit time and  calculate the duration
        - use the vechicle type + duration + pricing strategy and calculate the amount
        - record the payment
        - tell the lot to free the spot

        return the amount charged
        '''

        ticket.exit_time = datetime.now()

        # calculate the amount 
        amount = self.pricing.calculate(ticket) # pricing stategy is being used
        
        ticket.amount_paid = amount


        self.lot.vacate(ticket)

        print(f"[Gate {self.gate_id}] Ticket {ticket.id} closed. "
              f"Amount: ₹{amount:.2f}")

        return amount
        
        
    
            

        

## Pricing Strategy

In [65]:
# DON'T DO THIS  ---> violate the OCP principle
def process(self, ticket):
    duration_hours = ...
    if isinstance(ticket.vehicle, Bike):
        amount = 10 * duration_hours
    elif isinstance(ticket.vehicle, Car):
        amount = 30 * duration_hours
    elif isinstance(ticket.vehicle, Truck):
        amount = 60 * duration_hours
    ...

In [86]:
import math
from abc import ABC, abstractmethod

'''
We will define an interface that will take a ticket and calculate amount based on the ticket. 


'''

class PricingStrategy(ABC):
    
    @abstractmethod
    def calculate(self, ticket:Ticket)->float:
        '''
        Compute the amount due for a ticket whose exit_time is set. 
        '''
        pass


class HourlyPricing(PricingStrategy):
    RATES: dict[type, float] = {
        Bike: 10.0,
        Car: 30.0,
        Truck: 60.0,
    }
    
    def calculate(self, ticket:Ticket):
        if ticket.exit_time is None:
            raise ValueError("Cannot price an unfinished ticket (no exit_time)")

        # Compute duration in hours (as a float).
        duration_seconds = (ticket.exit_time - ticket.entry_time).total_seconds()
        duration_hours = duration_seconds / 3600

        # Round UP to the next started hour.
        billable_hours = max(1, math.ceil(duration_hours))

        vehicle_class = type(ticket.vehicle)
        rate = self.RATES[vehicle_class]

        return rate * billable_hours

        
class FlatRatePricing(PricingStrategy):
    RATES: dict[type, float] = {
        Bike: 50.0,
        Car: 150.0,
        Truck: 300.0,
    }
    def calculate(self, ticket:Ticket):
        if ticket.exit_time is None:
            raise ValueError("Cannot price an unfinished ticket")
        return self.RATES[type(ticket.vehicle)]
    



## Display Board

    - Customer wants to know if the lot is already full. So display board will show the information. 
    - Display board will just display the information
    - How to sync the board with the floor:
        - Option A -> Polling ( display will ask the floor)
        - Option B -> Pushing ( floor will inform the display proactively ) --> [Observer Pattern]
    

In [67]:
# Abstract class for the floor observer
class FloorObserver(ABC):
    
    @abstractmethod
    def update(self, floor:Floor):
        """Called by Floor whenever its occupancy state changes."""
        pass

# concrete class

class DisplayBoard(FloorObserver):
    '''Shows real-time free-spot counts for one floor, broken down by size.'''

    def __init__(self, label:str = ""):
        self.label = label


    def update(self, floor:Floor):
        # Pull what we need from the floor.
        counts = floor.free_spots_by_size() # this will give size wise available slots in the floor

        prefix = f"[{self.label}] " if self.label else ""
        print(
            f"{prefix}Floor {floor.floor_number} free → "
            f"Small: {counts[SpotSize.SMALL]}, "
            f"Medium: {counts[SpotSize.MEDIUM]}, "
            f"Large: {counts[SpotSize.LARGE]}"
        )
        





## Parking Lot

    - Attributes
            - collection of floors
            - find a floor that can park the vechicle
            - know about the active tickets ( so that we can vacate )
            

In [90]:
class ParkingLot:


    def __init__(self, floors: list[Floor]):
        if not floors:
            raise ValueError("ParkingLot must have at least one floor")
        self.floors = floors

        self.active_tickets: dict[str, Ticket] = {}


    def park(self, vehicle:Vehicle):
        '''
        - check every floor for available space
        - if space is available, then issue the ticket
        
        '''

        for floor in self.floors:
            spot = floor.park(vehicle)
            if spot is not None:
                ticket = Ticket(vehicle = vehicle, spot = spot, floor_number=floor.floor_number)
                self.active_tickets[ticket.id] = ticket
                return ticket
                
        # All floors full for this vehicle's size category.
        return None
    
    def vacate(self, ticket:Ticket):
        '''
        Free the spot associated with this ticket and remove
        the ticket from the active set.
        '''

        floor = self.floors[ticket.floor_number]
        floor.vacate(ticket.spot)

        # Remove from active set. .pop with a default avoids KeyError
        self.active_tickets.pop(ticket.id, None)

        
        


    

## demo and walkthrough of the parking lot
    - build a parking lot
    - create some gates and assign to the lot
    - admit a vehicle and see whether we can get a ticket or not 
    - trace the logic through the different component

In [94]:
def build_sample_lot(num_floors: int = 3) -> ParkingLot:
    floors = []

    for floor_num in range(num_floors):
        spots = []
        spots += [
            ParkingSpot(f"F{floor_num}-S{i}", SpotSize.SMALL)
            for i in range(5)
        ]

        spots += [
            ParkingSpot(f"F{floor_num}-M{i}", SpotSize.MEDIUM)
            for i in range(10)
        ]

        spots += [
            ParkingSpot(f"F{floor_num}-L{i}", SpotSize.LARGE)
            for i in range(3)
        ]


        floor = Floor(floor_num, spots)
        floor.attach(DisplayBoard(label=f"Display-F{floor_num}"))

        floors.append(floor)

    return ParkingLot(floors)
        
    

In [103]:
lot = build_sample_lot(num_floors=2)
entry = EntryGate("E1", lot)
exit_gate = ExitGate("X1", lot, HourlyPricing())


## bike arriving
bike = Bike("KA-01-B-1234")
ticket = entry.admit(bike)



[Display-F0] Floor 0 free → Small: 4, Medium: 10, Large: 3
[Gate E1] Issued ticket a196071b-a8ab-4d85-9248-b6d09bc35689 to Bike(KA-01-B-1234)


In [104]:
car = Car("DL-01-B-1234")
ticket = entry.admit(car)

[Display-F0] Floor 0 free → Small: 4, Medium: 9, Large: 3
[Gate E1] Issued ticket 98a69d9f-be36-40d1-9052-650a14752122 to Car(DL-01-B-1234)


In [105]:
ticket

Ticket(vehicle=Car(DL-01-B-1234), spot=Spot(F0-M0, size=MEDIUM, occupied by DL-01-B-1234), floor_number=0, entry_time=datetime.datetime(2026, 7, 19, 8, 59, 51, 831233), exit_time=None, amount_paid=None, id='98a69d9f-be36-40d1-9052-650a14752122')

In [106]:
bike = Bike("KA-01-A-32538")
ticket = entry.admit(bike)

[Display-F0] Floor 0 free → Small: 3, Medium: 9, Large: 3
[Gate E1] Issued ticket 519c2f38-e2a3-4f4f-98e7-2df52a162485 to Bike(KA-01-A-32538)


In [107]:
truck = Truck("KA-01-T-9999")
ticket3 = entry.admit(truck)

[Display-F0] Floor 0 free → Small: 3, Medium: 9, Large: 2
[Gate E1] Issued ticket 8eed43ba-9bca-4930-9637-e37eb7254574 to Truck(KA-01-T-9999)


In [108]:
ticket

Ticket(vehicle=Bike(KA-01-A-32538), spot=Spot(F0-S1, size=SMALL, occupied by KA-01-A-32538), floor_number=0, entry_time=datetime.datetime(2026, 7, 19, 9, 0, 2, 911003), exit_time=None, amount_paid=None, id='519c2f38-e2a3-4f4f-98e7-2df52a162485')

In [109]:
ticket3

Ticket(vehicle=Truck(KA-01-T-9999), spot=Spot(F0-L0, size=LARGE, occupied by KA-01-T-9999), floor_number=0, entry_time=datetime.datetime(2026, 7, 19, 9, 0, 9, 351772), exit_time=None, amount_paid=None, id='8eed43ba-9bca-4930-9637-e37eb7254574')

In [110]:
# the truck is now exiting

amount = exit_gate.process(ticket)

[Display-F0] Floor 0 free → Small: 4, Medium: 9, Large: 2
[Gate X1] Ticket 519c2f38-e2a3-4f4f-98e7-2df52a162485 closed. Amount: ₹10.00


In [111]:
ticket

Ticket(vehicle=Bike(KA-01-A-32538), spot=Spot(F0-S1, size=SMALL, free), floor_number=0, entry_time=datetime.datetime(2026, 7, 19, 9, 0, 2, 911003), exit_time=datetime.datetime(2026, 7, 19, 9, 0, 43, 601651), amount_paid=10.0, id='519c2f38-e2a3-4f4f-98e7-2df52a162485')

In [34]:
lot.floors

[Floor(0, 18 spots), Floor(1, 18 spots)]

## Edge  Case

    - Lot is full
    - vehicle has a size for which the spot is not available
    - ticket is presented two times
    - display board issues --( display board not attached )

In [115]:
for i in range(70):
    entry.admit(Bike(f"BIKE-{i}"))

## Follow up questions / Future scope of work 
    - How would you handle multiple gates parking simultaneously?
    - What about reserved spots like handicapped, electric, VIP?
    - How would you support different pricing on weekends or holidays
    - What if the lot has 100,000 spots?
    - How would you persist this?
    - How would you persist this?

## Summary

    - Design Decisions Have Reasons. State Them.
    - Defer, Don't Pre-Solve
    - Patterns are used as tools only when needed
    - Comments Earn Their Keep
    - Walk Through Your Design Before You Trust It